# Curs: Regularizarea în regresie – Ridge și Lasso

**Prerechizite.** Regresia liniară (OLS), împărțirea train/test, validarea încrucișată și noțiunea de multicoliniaritate (LP5, LP6).

**Obiective.** După acest curs veți putea:
- explica *de ce* regresia liniară obișnuită eșuează în unele situații (variabile corelate, multe variabile față de puține observații);
- scrie și interpreta funcțiile-obiectiv Ridge, Lasso și ElasticNet și explica diferența dintre ele (de ce Lasso face selecție de variabile, iar Ridge nu);
- alege hiperparametrul $\alpha$ corect (validare încrucișată, grila logaritmică, marginea grilei, regula „o eroare standard”);
- aplica și interpreta aceste modele într-un pipeline corect (standardizare, fără *leakage*);
- recunoaște limitele regularizării.

**Structura.** Teorie și cod ruleabil în același notebook. Celulele ✍️ **Exercițiu** sunt de rezolvat individual (⭐ simplu · ⭐⭐ mediu · ⭐⭐⭐ provocare). Soluțiile sunt într-un notebook separat.

| Partea | Conținut |
|---|---|
| 1 | Problema: de ce nu ajunge OLS și ideea regularizării |
| 2 | Ridge |
| 3 | Lasso |
| 4 | ElasticNet |
| 5 | Demonstrații pe date simulate |
| 6 | Compromisul bias–varianță și alegerea lui $\alpha$ |
| 7 | Aplicație completă |
| 8 | Capcane și rezumat |
| 9 | Exerciții și întrebări de verificare |


In [ ]:
# Dacă lipsesc pachetele (local), dezcomentați linia următoare. În Google Colab sunt deja instalate.
# !pip install -q scikit-learn seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.linear_model import (ElasticNet, ElasticNetCV, Lasso, LassoCV, LinearRegression,
                                  Ridge, RidgeCV, lasso_path)
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
# Setul "diabetes": 442 pacienți, 10 variabile; ținta = progresia bolii după un an (valoare numerică)
X, y = load_diabetes(as_frame=True, scaled=False, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Pentru GRAFICELE de mai jos (nu pentru evaluarea performanței) standardizăm pe tot setul
Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
yc = y - y.mean()
X.head()

## 1. De ce nu ajunge regresia liniară obișnuită (OLS)?

OLS alege coeficienții care minimizează suma pătratelor erorilor:

$$\hat\beta_{OLS} = \arg\min_\beta \sum_{i=1}^n (y_i - x_i^\top \beta)^2 = (X^\top X)^{-1} X^\top y$$

Soluția există și este stabilă doar dacă matricea $X^\top X$ se poate inversa „bine”. Două situații frecvente o strică:

1. **Variabile puternic corelate (multicoliniaritate).** $X^\top X$ este aproape singulară; coeficienții devin extrem de sensibili la mici schimbări ale datelor.
2. **Prea multe variabile față de observații** ($p$ mare, $n$ mic). Modelul are destulă libertate ca să „memoreze” zgomotul (*supraînvățare*). Dacă $p \ge n$, $X^\top X$ nu mai este inversabilă și nu există o soluție unică.

### 1.1 Ideea: penalizăm coeficienții mari

Pe lângă eroarea de potrivire, adăugăm o **penalizare** pe mărimea coeficienților, înmulțită cu $\alpha$:

$$\hat\beta = \arg\min_\beta \;\underbrace{\sum_{i=1}^n (y_i - \beta_0 - x_i^\top\beta)^2}_{\text{eroarea}} + \alpha \cdot \text{penalizare}(\beta)$$

- **Ridge (L2):** penalizare $\sum_j \beta_j^2$; echivalent, minimizăm eroarea cu restricția $\sum_j \beta_j^2 \le t$ (un „buget" pentru mărimea coeficienților).
- **Lasso (L1):** penalizare $\sum_j |\beta_j|$; echivalent, restricția $\sum_j |\beta_j| \le t$ (în `scikit-learn` eroarea este împărțită la $2n$).
- **$\alpha = 0$** înseamnă OLS (fără penalizare); **$\alpha$ mare** înseamnă coeficienți mici: un model mai simplu și mai stabil, în schimbul unui mic bias. $\alpha$ se alege din date, prin validare încrucișată (secțiunea 6).

Ridge și Lasso diferă prin forma penalizării: Ridge micșorează coeficienții, Lasso îi poate și anula. Le studiem pe rând în secțiunile 2–4; secțiunea 5 le pune la lucru pe date simulate, unde cunoaștem adevărul.

## 2. Regresia Ridge (penalizare L2)

**Funcția-obiectiv** (convenția scikit-learn):

$$\hat\beta_{ridge} = \arg\min_\beta \;\underbrace{\sum_{i=1}^n (y_i - \beta_0 - x_i^\top\beta)^2}_{\text{eroarea}} + \alpha \underbrace{\sum_{j=1}^p \beta_j^2}_{\text{penalizare L2}}$$

Echivalent: minimizăm eroarea cu **restricția** $\sum_j \beta_j^2 \le t$ (un „buget” pentru mărimea coeficienților; $\alpha$ mare $\leftrightarrow$ $t$ mic).

**Soluția în formă închisă:**

$$\hat\beta_{ridge} = (X^\top X + \alpha I)^{-1} X^\top y$$

Adăugarea lui $\alpha I$ face matricea **mereu inversabilă** pentru $\alpha > 0$, chiar și cu variabile corelate sau cu $p > n$. Aceasta este explicația matematică a stabilității pe care o demonstrăm pe date simulate în secțiunea 5.

**Proprietăți importante:**
- Micșorează toți coeficienții spre 0, dar **nu îi face exact 0** (nu face selecție de variabile).
- Cu variabile ortogonale și standardizate, $\hat\beta_{ridge,j} = \hat\beta_{OLS,j} \cdot \frac{n}{n+\alpha}$: o micșorare proporțională, aceeași pentru toți.
- Variabilele corelate își **împart** efectul (comportamentul „grupat” din demonstrația 5.1).
- **Interceptul $\beta_0$ nu se penalizează.** Altfel, rezultatul ar depinde de unde așezăm originea țintei.
- **Variabilele trebuie standardizate.** Penalizarea $\sum \beta_j^2$ depinde de unitatea de măsură: o variabilă în metri ar fi penalizată altfel decât aceeași variabilă în milimetri.

### 2.1 Verificare numerică a formulei

Calculăm coeficienții Ridge direct din formulă și îi comparăm cu `sklearn.linear_model.Ridge`:

In [ ]:
alpha = 10
p_ = Xs.shape[1]
# Soluția în formă închisă (date deja centrate, deci interceptul nu intră în calcul)
beta_manual = np.linalg.solve(Xs.T.values @ Xs.values + alpha * np.eye(p_), Xs.T.values @ yc.values)
beta_sklearn = Ridge(alpha=alpha).fit(Xs, y).coef_

print(pd.DataFrame({"formula manuală": beta_manual, "sklearn Ridge": beta_sklearn}, index=X.columns).round(4))
print("Coincid?", np.allclose(beta_manual, beta_sklearn))


### 2.2 Direcții: ce sunt și de ce Ridge funcționează (slide-urile 9 și 11)

**Ce este o „direcție"?** Nu este o variabilă, ci o **combinație de variabile**, un „ax nou". Pentru două variabile aproape identice:

- direcția „sumă": $z_1=(x_1+x_2)/\sqrt2$ (datele variază mult pe ea);
- direcția „diferență": $z_2=(x_1-x_2)/\sqrt2$ (datele aproape nu variază: $x_1\approx x_2$).

Cele două direcții sunt perpendiculare. **Valoarea proprie** $\lambda$ a unei direcții este suma pătratelor datelor pe ea: cât variază datele în acea direcție. La $p$ variabile există $p$ direcții perpendiculare. Dacă variabilele ar fi necorelate și standardizate, direcțiile ar coincide cu variabilele, iar $\lambda=n$ pentru toate.


In [ ]:

rho0 = 0.9975
rng_dir = np.random.default_rng(0)                         # generator propriu
Xd = rng_dir.multivariate_normal([0, 0], [[1, rho0], [rho0, 1]], size=100)
Xd = Xd - Xd.mean(axis=0)
U = np.array([[1, 1], [1, -1]]) / np.sqrt(2)               # coloanele = direcțiile „sumă” și „diferență”
Z = Xd @ U                                                  # coordonatele punctelor pe cele două direcții (z₁, z₂)
lam_dir = (Z ** 2).sum(axis=0)                              # λ = suma pătratelor datelor pe direcție

print(f"corelația x₁–x₂ = {np.corrcoef(Xd.T)[0, 1]:.4f}  |  corelația z₁–z₂ = {np.corrcoef(Z.T)[0, 1]:.3f} (≈ 0: direcții aproape necorelate)")
print(f"λ (sumă) = {lam_dir[0]:.1f}   |   λ (diferență) = {lam_dir[1]:.2f}   (teoretic ≈ 200 și 0,25)")

fig, ax = plt.subplots(figsize=(6.6, 5.8))
ax.scatter(Xd[:, 0], Xd[:, 1], s=26, color="#0F766E", alpha=0.55, linewidths=0, zorder=2)
for u in (U[:, 0], U[:, 1]):
    ax.plot([-4 * u[0], 4 * u[0]], [-4 * u[1], 4 * u[1]], color="#CBD5E1", ls="--", lw=1.2, zorder=1)
s1_, s2_ = 2 * np.sqrt(lam_dir / 100)                      # lungimea săgeților = 2 · deviația standard pe direcție
ax.annotate("", xy=(s1_ * U[0, 0], s1_ * U[1, 0]), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#E4572E", lw=3), zorder=4)
ax.annotate("", xy=(s2_ * U[0, 1], s2_ * U[1, 1]), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#1B2A35", lw=3), zorder=4)
ax.text(1.5, 2.6, "direcția „sumă”\ndatele variază mult", color="#E4572E", fontsize=12, fontweight="bold")
ax.annotate("direcția „diferență”\ndatele aproape nu variază", xy=(0.12, -0.12), xytext=(0.9, -2.9), color="#1B2A35", fontsize=12,
            fontweight="bold", arrowprops=dict(arrowstyle="-", color="#1B2A35", lw=1.3))
ax.set(xlim=(-4, 4), ylim=(-4, 4), aspect="equal", xlabel="x₁", ylabel="x₂", title="Două variabile aproape identice: două direcții")
plt.show()



**De ce coeficientul Ridge este coeficientul OLS înmulțit cu $\lambda_i/(\lambda_i+\alpha)$** (slide 11). Demonstrație în patru pași:

1. **Rotim variabilele în direcții.** Din $X^\top X=Q\Lambda Q^\top$ ($Q$ rotație, $\Lambda$ diagonală) definim $Z=XQ$ și $\beta=Q\gamma$. Atunci $X\beta=Z\gamma$ (aceleași predicții) și $Z^\top Z=\Lambda$ (variabile noi necorelate).
2. **Penalizarea nu se schimbă:** o rotație păstrează lungimile, deci $\|\beta\|^2=\|\gamma\|^2$.
3. **Problema se desparte pe direcții:**
$$\|y-Z\gamma\|^2+\alpha\|\gamma\|^2=\sum_i\left(\lambda_i\gamma_i^2-2\gamma_i z_i^\top y+\alpha\gamma_i^2\right)+\text{const.}$$
   Fiecare $\gamma_i$ se minimizează independent: $2(\lambda_i+\alpha)\gamma_i-2z_i^\top y=0$, deci $\hat\gamma_i^{ridge}=\dfrac{z_i^\top y}{\lambda_i+\alpha}$.
4. **Comparăm cu OLS** ($\alpha=0$): $\hat\gamma_i^{OLS}=\dfrac{z_i^\top y}{\lambda_i}$, deci
$$\frac{\hat\gamma_i^{ridge}}{\hat\gamma_i^{OLS}}=\frac{\lambda_i}{\lambda_i+\alpha}.$$

*Intuiție:* $\lambda_i$ este informația din date pe direcția $i$, iar $\alpha$ este informație adăugată artificial („coeficientul este 0"). Dacă $\lambda_i\gg\alpha$, datele domină; dacă $\lambda_i\ll\alpha$, penalizarea domină.


### 2.3 Ridge pe `diabetes`: direcții și micșorare inegală (slide-urile 13, 14 și 16)

*Celulele folosesc `X`, `y` și `Xs` (variabilele standardizate) din celula de date; nu consumă `rng`.*

**Verificarea factorului pe date reale.** Calculăm direcțiile lui $X^\top X$ pentru cele 10 variabile și verificăm că, în coordonatele direcțiilor, coeficientul Ridge este coeficientul OLS înmulțit cu $\lambda/(\lambda+\alpha)$.


In [ ]:

lam_d, Q_d = np.linalg.eigh(Xs.T.values @ Xs.values)       # λ crescător; coloanele lui Q = direcțiile
alpha_d = 10
b_ols_d = LinearRegression().fit(Xs, y).coef_
b_rdg_d = Ridge(alpha=alpha_d).fit(Xs, y).coef_
g_ols_d, g_rdg_d = Q_d.T @ b_ols_d, Q_d.T @ b_rdg_d        # coeficienții în coordonatele direcțiilor: γ = Qᵀβ

dir_tab = pd.DataFrame({"λ": lam_d, "γ̂ OLS": g_ols_d, "γ̂ Ridge": g_rdg_d,
                        "raport Ridge/OLS": g_rdg_d / g_ols_d, "λ/(λ+α)": lam_d / (lam_d + alpha_d)},
                       index=[f"direcția {i + 1}" for i in range(len(lam_d))])
print("Raportul Ridge/OLS coincide cu λ/(λ+α)?", np.allclose(g_rdg_d / g_ols_d, lam_d / (lam_d + alpha_d)))
dir_tab.round(3)



**OLS față de Ridge (slide 13).** Aceleași coeficienți, înapoi în coordonatele variabilelor: nu se micșorează toți la fel.


In [ ]:

vs_tab = pd.DataFrame({"OLS": b_ols_d, "Ridge (α=10)": b_rdg_d}, index=X.columns)
vs_tab["cât rămâne (%)"] = 100 * vs_tab["Ridge (α=10)"] / vs_tab["OLS"]
vs_tab.round(2)



`bmi`, `bp`, `sex` rămân aproape neschimbați (96–99,5%); `s1` și `s2` (corelate la 0,90) rămân la ≈ 30% și ≈ 8%; `s3` își schimbă semnul, iar `s6` chiar crește ușor: cu variabile corelate, un coeficient individual se poate mișca în orice direcție (norma totală scade mereu).

**De ce se micșorează inegal: valorile proprii (slide 14).**


In [ ]:

factor_d = lam_d / (lam_d + alpha_d)
fig, ax = plt.subplots(figsize=(9, 4.5))
cul = ["#E4572E", "#D99A00"] + ["#0F766E"] * (len(lam_d) - 2)
etich = [f"{l:.1f}" if l < 100 else f"{l:.0f}" for l in lam_d]
ax.bar(etich, factor_d, color=cul)
for i_, v_ in enumerate(factor_d):
    ax.text(i_, v_ + 0.02, f"{v_:.2f}", ha="center")
ax.set(ylim=(0, 1.1), xlabel="valoarea proprie λ a direcției (XᵀX)", ylabel="λ / (λ + 10)",
       title="Factorul de micșorare pe cele 10 direcții (α = 10)")
plt.show()

for k_ in (0, 1):                                            # cele două direcții slabe: ce variabile le formează?
    idx_ = np.argsort(-np.abs(Q_d[:, k_]))[:3]
    print(f"direcția cu λ = {lam_d[k_]:.1f} (factor {factor_d[k_]:.3f}):",
          ", ".join(f"{X.columns[i_]} {Q_d[i_, k_]:+.2f}" for i_ in idx_))



Două direcții sunt slabe ($\lambda\approx3{,}8$ și $34{,}6$; restul $\ge190$) și vin din variabilele corelate: prima este în principal `s1` față de `s2`, a doua `s4` cu `s3` și `s2`. Cu $\alpha=10$ ele se micșorează la 27,5% și 77,6%, restul rămân la $\ge95\%$. De aceea `s1`–`s4` se mișcă mult, iar `bmi` și `bp` aproape deloc.

**Același $\alpha$, alt $n$ (slide 16).** Pentru variabile standardizate și ortogonale, factorul este $n/(n+\alpha)$ (direcții cu $\lambda=n$).


In [ ]:

ns = [20, 50, 100, 442, 1000, 8000]
fac_n = pd.Series({f"n = {n_}": n_ / (n_ + alpha_d) for n_ in ns})
ax = fac_n.plot.bar(figsize=(8, 4), color=["#E4572E", "#E4572E", "#D99A00", "#0F766E", "#0F766E", "#0F766E"], rot=0,
                    title="Factorul n/(n + α) pentru α = 10")
for i_, v_ in enumerate(fac_n):
    ax.text(i_, v_ + 0.015, f"{v_:.3f}", ha="center")
ax.set_ylim(0, 1.1)
plt.show()
print(f"α = n  →  factor = {1000 / (1000 + 1000):.1f}   (reper: coeficienții la jumătate)")



La $n=50$ coeficienții se micșorează cu ≈ 17%, la $n=442$ (`diabetes`) cu ≈ 2% (dar `s2` rămâne la 8%, din cauza direcțiilor slabe), la $n=8.000$ (LP6) cu ≈ 0,1%: practic OLS. Regula e pentru variabile ortogonale; cu variabile corelate, direcțiile slabe se micșorează mult mai mult.

### 2.4 Calea Ridge (*regularization path*) (slide 17)

Pe `diabetes`, urmărim cum se schimbă coeficienții când $\alpha$ crește de la foarte mic (≈ OLS) la foarte mare.

In [ ]:
alphas_r = np.logspace(-2, 4, 60)
path_r = np.array([Ridge(alpha=a).fit(Xs, y).coef_ for a in alphas_r])

fig, ax = plt.subplots(figsize=(9, 5))
for j, col in enumerate(X.columns):
    ax.plot(alphas_r, path_r[:, j], label=col)
ax.axhline(0, color="black", lw=0.8)
ax.set(xscale="log", xlabel="alpha (scară logaritmică)", ylabel="coeficient (variabile standardizate)",
       title="Calea Ridge: coeficienții se micșorează, dar nu devin exact 0")
ax.legend(ncol=2, fontsize=8)
plt.show()

La $\alpha$ mic (stânga) coeficienții sunt cei OLS; unii au valori mari și semne opuse (variabilele `s1`, `s2`, `s3`, `s4`, `s5` sunt puternic corelate între ele). Pe măsură ce $\alpha$ crește, coeficienții se apropie lent de 0, **fără să devină exact 0**; mai întâi se calmează perechile corelate, apoi toți coeficienții se micșorează împreună.

### 2.5 Cum se citește calea Ridge (slide 18)

Valorile coeficienților la $\alpha$ mic față de $\alpha$ mediu.

In [ ]:

cale_tab = pd.DataFrame({"α = 0,01": Ridge(alpha=0.01).fit(Xs, y).coef_,
                         "α = 100": Ridge(alpha=100).fit(Xs, y).coef_}, index=X.columns).loc[["bmi", "bp", "s5", "s1", "s2", "s3"]]
a_, b_ = cale_tab["α = 0,01"], cale_tab["α = 100"]
cale_tab["schimbare"] = np.where(np.sign(a_) != np.sign(b_), "semn schimbat",
                                 ((b_.abs() - a_.abs()) / a_.abs() * 100).round(0).astype(int).astype(str) + "%")
cale_tab.round(1)



Coeficienții stabili (`bmi`, `bp`, `s5`) se micșorează lent; cei instabili (`s1`, `s2`, `s3`) se mișcă mult, iar `s2` și `s3` își schimbă semnul. La $\alpha$ mic avem practic OLS, la $\alpha$ mediu ($\approx1$–$100$) se „calmează" perechile corelate (direcțiile slabe, $\lambda\approx4$ și $35$), iar la $\alpha$ mare ($\ge1.000$) toți coeficienții tind spre 0 (fără să fie exact 0).


## 3. Regresia Lasso (penalizare L1)

**Funcția-obiectiv** (convenția scikit-learn; atenție la factorul $\frac{1}{2n}$, diferit de Ridge!):

$$\hat\beta_{lasso} = \arg\min_\beta \;\frac{1}{2n}\sum_{i=1}^n (y_i - \beta_0 - x_i^\top\beta)^2 + \alpha \underbrace{\sum_{j=1}^p |\beta_j|}_{\text{penalizare L1}}$$

(*Lasso* = *Least Absolute Shrinkage and Selection Operator*.) Penalizarea cu valoare absolută nu este derivabilă în 0, de aceea **nu există formulă închisă**; scikit-learn folosește *coordinate descent* (se actualizează câte un coeficient pe rând).

> **Atenție la convenții.** Din cauza factorului $\frac{1}{2n}$ la Lasso și a lipsei lui la Ridge, valorile lui `alpha` din cele două clase **nu sunt comparabile** direct.

### 3.1 De ce face Lasso selecție de variabile? Două explicații

**(a) Algebric: soft-thresholding.** Cu variabile ortonormale, soluțiile sunt:
- Ridge: $\hat\beta_j = \hat\beta_{OLS,j} / (1+\lambda)$ – micșorare proporțională, niciodată 0;
- Lasso: $\hat\beta_j = \text{sign}(\hat\beta_{OLS,j}) \cdot \max(|\hat\beta_{OLS,j}| - \lambda, 0)$ – scade coeficientul cu o **valoare fixă** și, dacă ar trece de 0, îl oprește la **exact 0**.

In [ ]:
lam = 1.0
b = np.linspace(-3, 3, 400)                       # coeficientul OLS
ridge_b = b / (1 + lam)                           # Ridge: micșorare proporțională
lasso_b = np.sign(b) * np.maximum(np.abs(b) - lam, 0)   # Lasso: soft-thresholding

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(b, b, "k--", label="OLS (fără penalizare)")
ax.plot(b, ridge_b, label="Ridge")
ax.plot(b, lasso_b, label="Lasso")
ax.set(xlabel="coeficientul OLS", ylabel="coeficientul penalizat", title=f"Efectul penalizării (λ = {lam})")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.legend()
plt.show()

Pentru Lasso, orice coeficient OLS cu $|\hat\beta| \le \lambda$ devine exact 0 („zona moartă”), pe când Ridge nu anulează nimic.

**(b) Geometric.** Problema cu restricție: cele două elipse sunt curbele de nivel ale erorii (centrul = soluția OLS); zona colorată este „bugetul” permis. Soluția este primul punct în care elipsa atinge zona.

In [ ]:
# Contururile erorii (elipse) în jurul soluției OLS și regiunile permise de penalizare
A = np.array([[1.0, 0.5], [0.5, 0.5]])
b_ols = np.array([2.0, 1.0])
t = 1.5                                            # „bugetul” penalizării

g1, g2 = np.meshgrid(np.linspace(-1.8, 2.8, 400), np.linspace(-1.8, 2.0, 400))
D = np.stack([g1 - b_ols[0], g2 - b_ols[1]], axis=-1)
rss = np.einsum("...i,ij,...j->...", D, A, D)

th = np.linspace(0, 2 * np.pi, 4001)
circle = np.c_[t * np.cos(th), t * np.sin(th)]
edges = []
for s1, s2 in [(1, 1), (1, -1), (-1, 1), (-1, -1)]:
    u = np.linspace(0, t, 2001)
    edges.append(np.c_[s1 * u, s2 * (t - u)])
diamond = np.vstack(edges)

def best_point(pts):
    d = pts - b_ols
    return pts[np.argmin(np.einsum("ni,ij,nj->n", d, A, d))]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, title, shape, patch in [(axes[0], "Ridge: |β|² ≤ t (cerc)", circle, plt.Circle((0, 0), t, alpha=0.15)),
                                (axes[1], "Lasso: |β₁| + |β₂| ≤ t (romb)", diamond,
                                 plt.Polygon([[t, 0], [0, t], [-t, 0], [0, -t]], alpha=0.15))]:
    ax.contour(g1, g2, rss, levels=[0.1, 0.3, 0.6, 1.0, 1.5], colors="tab:blue", linewidths=0.9)
    ax.add_patch(patch)
    sol = best_point(shape)
    ax.scatter(*b_ols, color="black", zorder=5); ax.annotate("OLS", b_ols + 0.05)
    ax.scatter(*sol, color="red", zorder=5, s=60); ax.annotate("soluția penalizată", sol + [-0.1, -0.4], color="red")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set(xlim=(-1.8, 2.8), ylim=(-1.8, 2.0), aspect="equal", xlabel="β₁", ylabel="β₂", title=title)
plt.tight_layout()
plt.show()

Zona Ridge este un **cerc** (fără colțuri): punctul de atingere are, în general, ambele coordonate nenule. Zona Lasso este un **romb** cu colțuri pe axe: elipsa atinge cel mai des un colț, unde o coordonată este **exact 0**. În dimensiuni mari, rombul (hiper-octaedrul) are tot mai multe colțuri și muchii, deci tot mai multe șanse ca multe coordonate să fie 0.

### 3.2 Calea Lasso

In [ ]:
alphas_l, coefs_l, _ = lasso_path(Xs, yc, alphas=np.logspace(-2, 2, 100))

fig, ax = plt.subplots(figsize=(9, 5))
for j, col in enumerate(X.columns):
    ax.plot(alphas_l, coefs_l[j], label=col)
ax.axhline(0, color="black", lw=0.8)
ax.set(xscale="log", xlabel="alpha (scară logaritmică)", ylabel="coeficient (variabile standardizate)",
       title="Calea Lasso: variabilele „intră” pe rând în model pe măsură ce alpha scade")
ax.legend(ncol=2, fontsize=8)
plt.show()

# Ordinea în care variabilele devin nenule (de la alpha mare spre alpha mic)
first_nonzero = {col: alphas_l[np.nonzero(coefs_l[j])[0][0]] for j, col in enumerate(X.columns) if np.any(coefs_l[j])}
pd.Series(first_nonzero).sort_values(ascending=False).round(2).rename("alpha la care devine nenulă")

Citiți graficul de la dreapta spre stânga: la $\alpha$ mare toți coeficienții sunt 0 (modelul prezice doar media); pe măsură ce $\alpha$ scade, variabilele **intră pe rând** în model, începând cu cele mai importante. Valoarea de la care toți coeficienții sunt 0 este $\alpha_{max} = \max_j |x_j^\top y_c|/n$ (aici, ordinul zecilor, fiindcă ținta are o scară mare); de aceea grila pentru Lasso trebuie adaptată la scara lui $y$.

### 3.3 Limitele Lasso
- Între variabile **foarte corelate** alege una, aproape arbitrar (o vedem în demonstrația 5.1); selecția poate fi instabilă între eșantioane.
- Poate selecta cel mult $n$ variabile.
- Coeficienții rămași sunt **deplasați spre 0** (bias), deci nu sunt estimări „curate” ale efectului.

## 4. ElasticNet: combinația

$$\min_\beta \;\frac{1}{2n}\sum_i (y_i - x_i^\top\beta)^2 + \alpha\left[\rho \sum_j |\beta_j| + \frac{1-\rho}{2}\sum_j \beta_j^2\right], \qquad \rho = \texttt{l1\_ratio} \in [0,1]$$

- $\rho = 1$: Lasso pur; $\rho \to 0$: Ridge (aproape) pur.
- Păstrează avantajele ambelor: **selecție de variabile** (partea L1) și **stabilitate la variabile corelate**, cu efect „grupat” (partea L2): variabilele corelate tind să fie păstrate sau eliminate împreună.
- Are doi hiperparametri ($\alpha$ și $\rho$), ambii aleși prin validare încrucișată (`ElasticNetCV`).

| | Ridge | Lasso | ElasticNet |
|---|---|---|---|
| Penalizare | $\sum\beta_j^2$ | $\sum\lvert\beta_j\rvert$ | combinație |
| Coeficienți exact 0 | nu | da | da |
| Variabile corelate | le împarte efectul | alege una | le păstrează/elimină împreună |
| Formulă închisă | da | nu | nu |
| Când îl alegem | multe variabile cu efect mic; multicoliniaritate | credem că doar câteva variabile contează; vrem model rar | variabile corelate, dar vrem și selecție |

## 5. Demonstrații pe date simulate

Acum că știm cum funcționează Ridge, Lasso și ElasticNet, vedem ce fac pe date simulate, unde **cunoaștem adevărul**: cum se comportă coeficienții când variabilele sunt corelate (5.1), ce se întâmplă când avem prea puține observații (5.2) și cum depinde alegerea dintre Ridge și Lasso de structura problemei (5.3).

### 5.1 Coeficienți instabili (slide 27)

Două variabile $x_1, x_2$ aproape identice. În realitate $y = 1 \cdot x_1 + 1 \cdot x_2 + \text{zgomot}$. Repetăm experimentul pe 300 de eșantioane și ne uităm unde cad coeficienții estimați.

In [ ]:
def simulate(n=100):
    """Două variabile aproape identice (corelație ≈ 0,996); adevărul: y = 1·x1 + 1·x2 + zgomot."""
    z = rng.normal(size=n)
    x1 = z + 0.05 * rng.normal(size=n)
    x2 = z + 0.05 * rng.normal(size=n)
    return np.column_stack([x1, x2]), x1 + x2 + rng.normal(size=n)

Xd, _ = simulate()
print(f"Corelația dintre x1 și x2: {np.corrcoef(Xd.T)[0, 1]:.4f}")

coefs = {"OLS": [], "Ridge (α=10)": [], "Lasso (α=0,1)": []}
for _ in range(300):                      # 300 de eșantioane diferite din aceeași „lume”
    Xd, yd = simulate()
    coefs["OLS"].append(LinearRegression().fit(Xd, yd).coef_)
    coefs["Ridge (α=10)"].append(Ridge(alpha=10).fit(Xd, yd).coef_)
    coefs["Lasso (α=0,1)"].append(Lasso(alpha=0.1).fit(Xd, yd).coef_)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, (name, c) in zip(axes, coefs.items()):
    c = np.array(c)
    ax.scatter(c[:, 0], c[:, 1], alpha=0.4, s=15)
    ax.scatter([1], [1], color="red", marker="*", s=200, label="adevăr (1, 1)")
    ax.set(title=f"{name}\nstd(β1) = {c[:, 0].std():.2f}", xlabel="coeficientul lui x1")
axes[0].set_ylabel("coeficientul lui x2")
axes[0].legend()
plt.tight_layout()
plt.show()

**Ce vedem.**
- **OLS:** coeficienții se împrăștie enorm (deviație standard de ordinul 1,4, pentru un adevăr egal cu 1), dar sunt mereu pe o dreaptă $\beta_1 + \beta_2 \approx 2$. Modelul știe suma, dar nu poate împărți „meritul” între cele două variabile aproape identice. Predicțiile rămân bune; **interpretarea coeficienților individuali nu**.
- **Ridge:** coeficienții sunt strânși în jurul lui (≈ 0,96; ≈ 0,96), cu o împrăștiere de ≈ 0,06 (față de ≈ 1,4 la OLS): împarte meritul egal între variabilele corelate. Stabil, ușor „micșorat” față de adevăr.
- **Lasso:** în aproximativ jumătate din eșantioane anulează **una** dintre cele două variabile (punctele de pe axe), iar în celelalte le păstrează pe ambele; care dintre ele supraviețuiește diferă de la un eșantion la altul. Selecția între variabile aproape identice este arbitrară.


#### 5.1.1 De ce se clatină OLS? Dependența de corelația $\rho$ (slide 28)

Varianța coeficientului OLS este, aproximativ,

$$\mathrm{Var}(\hat\beta_1) \approx \frac{\sigma^2}{n\,(1-\rho^2)\,\mathrm{Var}(x_1)}$$

unde $\rho$ este corelația dintre $x_1$ și $x_2$. Când $\rho \to 1$, numitorul tinde la 0 și varianța explodează. Verificăm formula prin simulare: pentru fiecare $\rho$ repetăm de 300 de ori experimentul (100 de observații, $y = x_1 + x_2 + \text{zgomot}$) și calculăm deviația standard a lui $\hat\beta_1$.

*Celulele din 5.1 folosesc generatoare aleatoare proprii (`rng_rho`, `rng_s`, ...), deci nu schimbă rezultatele celorlalte celule.*


In [ ]:

rng_rho = np.random.default_rng(123)          # generator propriu: nu atinge `rng` global
rhos = [0.0, 0.5, 0.9, 0.99, 0.999]
n_obs, sigma_y, n_rep = 100, 1.0, 300

sweep = {"OLS (simulare)": [], "OLS (formulă)": [], "Lasso (α=0,1)": [], "Ridge (α=10)": []}
for r in rhos:
    cov = np.array([[1, r], [r, 1]])
    est = {"OLS (simulare)": [], "Ridge (α=10)": [], "Lasso (α=0,1)": []}
    for _ in range(n_rep):
        Xr = rng_rho.multivariate_normal([0, 0], cov, size=n_obs)
        yr = Xr.sum(axis=1) + sigma_y * rng_rho.normal(size=n_obs)
        est["OLS (simulare)"].append(LinearRegression().fit(Xr, yr).coef_[0])
        est["Ridge (α=10)"].append(Ridge(alpha=10).fit(Xr, yr).coef_[0])
        est["Lasso (α=0,1)"].append(Lasso(alpha=0.1).fit(Xr, yr).coef_[0])
    for k, v in est.items():
        sweep[k].append(np.std(v))
    sweep["OLS (formulă)"].append(sigma_y / np.sqrt(n_obs * (1 - r**2)))   # Var(x1) = 1

sweep_df = pd.DataFrame(sweep, index=[f"ρ = {r}" for r in rhos])

fig, ax = plt.subplots(figsize=(8, 4.8))
for col, style in [("OLS (simulare)", dict(color="#64748B", marker="o")), ("OLS (formulă)", dict(color="#1B2A35", marker="s", ls="--")),
                   ("Lasso (α=0,1)", dict(color="#E4572E", marker="o")), ("Ridge (α=10)", dict(color="#0F766E", marker="o"))]:
    ax.plot([str(r) for r in rhos], sweep_df[col], label=col, lw=2.5, **style)
ax.set(xlabel="corelația ρ dintre x₁ și x₂", ylabel="std(β₁) pe 300 de eșantioane",
       title="Deviația standard a lui β₁ în funcție de corelația ρ")
ax.legend()
plt.show()
sweep_df.round(3)


In [ ]:

# Valorile proprii ale XᵀX (date centrate): direcția „sumă” și direcția „diferență”, pentru ρ = 0,9975
rho0 = 0.9975
rng_e = np.random.default_rng(0)
Xe = rng_e.multivariate_normal([0, 0], [[1, rho0], [rho0, 1]], size=n_obs)
Xe = Xe - Xe.mean(axis=0)
lam_dif, lam_sum = np.linalg.eigvalsh(Xe.T @ Xe)

print(f"Teoretic: n(1+ρ) = {n_obs * (1 + rho0):.1f}   |   n(1−ρ) = {n_obs * (1 - rho0):.2f}")
print(f"Empiric (un eșantion): λ_sumă = {lam_sum:.1f}   |   λ_diferență = {lam_dif:.2f}")
print(f"Std(β₁) prevăzută de formulă: 1/√(n(1−ρ²)) = {1 / np.sqrt(n_obs * (1 - rho0**2)):.2f}")



**Ce vedem.** Formula se potrivește cu simularea (la $\rho = 0{,}999$: ≈ 2,24 față de ≈ 2,26). Intuitiv, $X^\top X$ are două direcții: una „sumă" (valoare proprie ≈ $n(1+\rho) \approx 200$), bine determinată, și una „diferență" ($n(1-\rho) \approx 0{,}25$), aproape nedeterminată. OLS știe suma $\beta_1 + \beta_2$, dar nu cum se împarte. Ridge rămâne stabil chiar și la corelații foarte mari, iar Lasso este la mijloc.

#### 5.1.2 Ridge și Lasso, definiția aplicată pe perechi de coeficienți (slide 29)

Penalizările, așa cum sunt definite:

- Ridge: $\alpha \sum_j \beta_j^2$, echivalent cu restricția $\sum_j \beta_j^2 \le t$;
- Lasso: $\alpha \sum_j |\beta_j|$, echivalent cu restricția $\sum_j |\beta_j| \le t$.

Luăm perechi de coeficienți **cu aceeași sumă** $\beta_1 + \beta_2 = 2$ (deci cu aproximativ aceeași eroare de potrivire, fiindcă $x_1 \approx x_2$) și calculăm cât costă fiecare pereche.


In [ ]:

pairs = [(1, 1), (2, 0), (0, 2), (3, -1)]
penalties = pd.DataFrame({
    "β₁ + β₂": [sum(p_) for p_ in pairs],
    "Ridge: Σβ²": [sum(b**2 for b in p_) for p_ in pairs],
    "Lasso: Σ|β|": [sum(abs(b) for b in p_) for p_ in pairs],
}, index=[str(p_) for p_ in pairs])
penalties.index.name = "(β₁, β₂)"
penalties



- **Ridge:** la aceeași sumă, perechea echilibrată $(1, 1)$ are cea mai mică penalizare (2, față de 4 la $(2, 0)$), deci Ridge o preferă: coeficienți egali.
- **Lasso:** toate perechile cu $\beta \ge 0$ și aceeași sumă au aceeași penalizare (2). Nicio preferință, deci decide zgomotul din eșantion.
- Rezultat: **Ridge împarte meritul; Lasso alege arbitrar.**

#### 5.1.3 Simularea coeficienților și rezumatul ei (slide 30)

Repetăm simularea din 5.1 (aceeași „lume", `seed` fix) și calculăm tot ce arată slide-ul 30: media și deviația standard a lui $\hat\beta_1$ și $\hat\beta_2$, **corelația dintre cei doi coeficienți**, suma medie și procentul de eșantioane în care Lasso a anulat exact o variabilă.


In [ ]:

rng_s = np.random.default_rng(42)             # generator propriu

def simulate_pair(n=100):
    z = rng_s.normal(size=n)
    x1 = z + 0.05 * rng_s.normal(size=n)
    x2 = z + 0.05 * rng_s.normal(size=n)
    return np.column_stack([x1, x2]), x1 + x2 + rng_s.normal(size=n)

est_s = {"OLS": [], "Ridge": [], "Lasso": []}
for _ in range(300):
    Xp, yp = simulate_pair()
    est_s["OLS"].append(LinearRegression().fit(Xp, yp).coef_)
    est_s["Ridge"].append(Ridge(alpha=10).fit(Xp, yp).coef_)
    est_s["Lasso"].append(Lasso(alpha=0.1).fit(Xp, yp).coef_)
est_s = {k: np.array(v) for k, v in est_s.items()}

summary = pd.DataFrame({
    k: {"media β₁": v[:, 0].mean(), "media β₂": v[:, 1].mean(),
        "std β₁": v[:, 0].std(), "std β₂": v[:, 1].std(),
        "corelația β₁–β₂": np.corrcoef(v.T)[0, 1],
        "media β₁+β₂": v.sum(axis=1).mean(), "std β₁+β₂": v.sum(axis=1).std(),
        "o variabilă anulată (%)": 100 * ((v == 0).sum(axis=1) == 1).mean()}
    for k, v in est_s.items()}).T

colors = {"OLS": "#64748B", "Ridge": "#0F766E", "Lasso": "#E4572E"}
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.8), sharex=True, sharey=True)
for ax, (name, v) in zip(axes, est_s.items()):
    s_ = summary.loc[name]
    ax.plot([-3, 5], [5, -3], color="#CBD5E1", ls="--", lw=1.2, zorder=1)            # β₁ + β₂ = 2
    ax.scatter(v[:, 0], v[:, 1], s=16, alpha=0.5, color=colors[name], linewidths=0, zorder=2)
    ax.scatter([1], [1], marker="*", s=300, color="black", zorder=5)                 # adevărul (1, 1)
    ax.set_xlim(-3, 5); ax.set_ylim(-3, 5); ax.set_aspect("equal")
    ax.set_title(f"{name}\nstd(β₁) = {s_['std β₁']:.2f} · std(β₂) = {s_['std β₂']:.2f}\ncorr(β₁, β₂) = {s_['corelația β₁–β₂']:.3f}", fontsize=11)
    ax.set_xlabel("coeficientul lui x₁ (β₁)")
axes[0].set_ylabel("coeficientul lui x₂ (β₂)")
plt.tight_layout()
plt.show()
summary.round(3)


#### 5.1.4 Verificare numerică pe exemplul sumă / diferență (slide 31)

Adevărul este $y=x_1+x_2=\sqrt2\,z_1+0\cdot z_2$, deci $\gamma_1=\sqrt2\approx1{,}41$ și $\gamma_2=0$. Cu $n=100$, $\sigma=1$, $\alpha=10$, formulele prevăd: $\lambda\approx(200{,}25;\ 0{,}25)$, deviația standard OLS $\sigma/\sqrt{\lambda_i}$ și deviația standard Ridge $\sigma\sqrt{\lambda_i}/(\lambda_i+\alpha)$. Comparăm cu cele 300 de eșantioane din 5.1.3 (`est_s`) și revenim la coeficienții originali prin $\hat\beta_1=(\hat\gamma_1+\hat\gamma_2)/\sqrt2$.


In [ ]:

n_obs_, sigma_, alpha_r = 100, 1.0, 10.0
lam_th = n_obs_ * np.array([1.0025 + 1.0, 1.0025 - 1.0])    # Var(x) = 1,0025 și Cov(x₁, x₂) = 1  →  λ = 200,25 și 0,25
fac_th = lam_th / (lam_th + alpha_r)                         # factorul λ/(λ+α) pe fiecare direcție
g_true = np.array([np.sqrt(2), 0.0])                         # adevărul în coordonatele direcțiilor

# formulele
th = {
    "OLS": {"std γ̂₁": sigma_ / np.sqrt(lam_th[0]), "std γ̂₂": sigma_ / np.sqrt(lam_th[1]), "media γ̂₁": g_true[0], "media γ̂₂": g_true[1]},
    "Ridge": {"std γ̂₁": sigma_ * np.sqrt(lam_th[0]) / (lam_th[0] + alpha_r), "std γ̂₂": sigma_ * np.sqrt(lam_th[1]) / (lam_th[1] + alpha_r),
              "media γ̂₁": g_true[0] * fac_th[0], "media γ̂₂": g_true[1] * fac_th[1]},
}
for k_ in th:
    th[k_]["std β̂₁"] = np.hypot(th[k_]["std γ̂₁"], th[k_]["std γ̂₂"]) / np.sqrt(2)
    th[k_]["media β̂₁"] = (th[k_]["media γ̂₁"] + th[k_]["media γ̂₂"]) / np.sqrt(2)

# simularea: γ̂ = Uᵀβ̂ pentru fiecare eșantion (U este simetrică, deci β̂ @ U)
ms = {}
for k_ in ("OLS", "Ridge"):
    G = est_s[k_] @ U
    ms[k_] = {"std γ̂₁": G[:, 0].std(), "std γ̂₂": G[:, 1].std(), "media γ̂₁": G[:, 0].mean(), "media γ̂₂": G[:, 1].mean(),
              "std β̂₁": est_s[k_][:, 0].std(), "media β̂₁": est_s[k_][:, 0].mean()}

print("λ =", lam_th.round(2), "| factor λ/(λ+α) =", fac_th.round(3))
comp = pd.DataFrame({f"{k_} {tip}": (th if tip == "formulă" else ms)[k_] for k_ in ("OLS", "Ridge") for tip in ("formulă", "simulare")})
comp.round(3)



Formulele se potrivesc cu simularea (de exemplu std($\hat\beta_1$): OLS ≈ 1,41 față de 1,42; Ridge ≈ 0,059 față de 0,057). Ce explică rezultatele:

- Pe direcția **diferență**, OLS are $\mathrm{std}(\hat\gamma_2)=\sigma/\sqrt{\lambda_2}=2$: foarte zgomotoasă, deși adevărul este 0. Ridge o reduce la ≈ 0,05 (factor 0,024).
- Zgomotul din $\hat\gamma_2$ intră cu **semne opuse** în $\hat\beta_1$ și $\hat\beta_2$: de aici anticorelarea (−0,998) și dreapta $\beta_1+\beta_2\approx2$ din figura de la 5.1.3.
- Pe direcția diferență adevărul este 0, deci micșorarea **nu costă bias**; pe direcția sumă factorul 0,952 introduce un mic bias (media $\hat\beta_1\approx0{,}95$).


#### 5.1.5 Concluzie: Ridge împarte, Lasso alege (slide 32)

In [ ]:

# Slide 32: comparația Ridge – Lasso, calculată din rezumat
o, r_, l = summary.loc["OLS"], summary.loc["Ridge"], summary.loc["Lasso"]
print(f"std(β₁): OLS {o['std β₁']:.3f} | Ridge {r_['std β₁']:.3f} (de ≈ {o['std β₁'] / r_['std β₁']:.0f} ori mai mică) | "
      f"Lasso {l['std β₁']:.3f} (de ≈ {o['std β₁'] / l['std β₁']:.1f} ori mai mică)")
print(f"corelația β₁–β₂: Ridge {r_['corelația β₁–β₂']:+.2f} | Lasso {l['corelația β₁–β₂']:+.2f} | OLS {o['corelația β₁–β₂']:+.2f}")
print(f"eșantioane cu o variabilă anulată: Ridge {r_['o variabilă anulată (%)']:.0f}% | Lasso {l['o variabilă anulată (%)']:.0f}%")
print(f"suma medie β₁+β₂ (std): OLS {o['media β₁+β₂']:.2f} ({o['std β₁+β₂']:.2f}) | Ridge {r_['media β₁+β₂']:.2f} | Lasso {l['media β₁+β₂']:.2f}")


- **Ridge:** std($\hat\beta_1$) de ≈ 25 de ori mai mică decât la OLS, corelație mică și pozitivă între $\hat\beta_1$ și $\hat\beta_2$, nicio variabilă anulată. Perechea echilibrată costă cel mai puțin, deci $\hat\beta_1 \approx \hat\beta_2 \approx 0{,}96$.
- **Lasso:** std($\hat\beta_1$) de ≈ 1,8 ori mai mică decât la OLS, coeficienți aproape perfect anticorelați (≈ −1, ca la OLS) și o variabilă anulată exact în ≈ 50% dintre eșantioane. Toate perechile cu aceeași sumă costă la fel, deci alege arbitrar.
- **În comun:** față de OLS ambele stabilizează coeficienții, iar suma $\hat\beta_1 + \hat\beta_2$ rămâne ≈ 2 (cu aceeași împrăștiere mică), deci predicțiile sunt bune. Diferă doar felul în care se împarte efectul între variabilele corelate.


### 5.2 Supraînvățare ($n$ mic, $p$ mare) (slide 33)

Avem 25 de variabile, dar **doar 3** influențează cu adevărat $y$; avem doar 30 de observații. Evaluăm pe un set de test mare (5.000 de observații).

In [ ]:
n_train, p = 30, 25
beta_true = np.zeros(p)
beta_true[:3] = [3, -2, 1.5]              # doar 3 variabile contează din 25

def make_data(n):
    Xn = rng.normal(size=(n, p))
    return Xn, Xn @ beta_true + rng.normal(size=n)

Xtr, ytr = make_data(n_train)
Xte, yte = make_data(5000)

fits = {
    "OLS": LinearRegression(),
    "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 60)),
    "Lasso": LassoCV(alphas=np.logspace(-3, 1, 60), cv=5, max_iter=50000),
}
rows = {}
for name, m in fits.items():
    m.fit(Xtr, ytr)
    rows[name] = {"R² train": m.score(Xtr, ytr), "R² test": m.score(Xte, yte),
                  "RMSE test": rmse(yte, m.predict(Xte)), "coef. ≠ 0": int((m.coef_ != 0).sum())}
pd.DataFrame(rows).T.round(3)

OLS se potrivește aproape perfect pe datele de antrenare (R² train ≈ 0,99), dar pe date noi scade mult (R² test ≈ 0,77 în rularea noastră): a învățat și zgomotul. Ridge și Lasso, antrenate pe *exact aceleași* 30 de observații, generalizează mai bine (R² test ≈ 0,82, respectiv ≈ 0,87). Lasso a anulat în plus aproape toate variabilele irelevante (în rularea noastră păstrează 8 din 25, deci 17 coeficienți sunt exact 0).


#### 5.2.1 De ce supraînvață OLS? Grade de libertate și maximul teoretic al lui $R^2$ (slide 34)

*Aceste celule folosesc modelele `fits` și datele `Xtr, ytr, Xte, yte, beta_true` din celula de mai sus; rulați-le după ea.*

OLS are $p + 1 = 26$ parametri (25 de coeficienți și interceptul) pentru $n = 30$ de observații, deci doar $30 - 26 = 4$ **grade de libertate** pentru zgomot. Mai mult, putem calcula cel mai bun $R^2$ posibil: varianța semnalului este $\sum_j \beta_j^2 = 3^2 + 2^2 + 1{,}5^2 = 15{,}25$, iar zgomotul are $\sigma^2 = 1$.


In [ ]:

n_params = p + 1                                     # 25 de coeficienți + interceptul
df_res = n_train - n_params                          # grade de libertate pentru zgomot
sigma2 = 1.0                                         # varianța zgomotului din make_data
var_signal = float(beta_true @ beta_true)            # variabilele sunt N(0, 1) independente
var_y = var_signal + sigma2
r2_max = var_signal / var_y                          # cel mai bun R² posibil pe date noi
r2_train_asteptat = 1 - df_res * sigma2 / (n_train * var_y)   # reziduurile OLS ≈ σ² · grade de libertate

print(f"Parametri: {n_params} | observații: {n_train} | grade de libertate: {df_res}")
print(f"Varianța semnalului = {var_signal:.2f}; maximul teoretic al lui R² = {var_signal:.2f}/{var_y:.2f} = {r2_max:.3f}")
print(f"R² pe antrenare așteptat pentru OLS ≈ 1 − {df_res}·{sigma2:.0f}/({n_train}·{var_y:.2f}) = {r2_train_asteptat:.3f}")

r2_bars = pd.Series({
    "Maxim teoretic": r2_max,
    "OLS antrenare": fits["OLS"].score(Xtr, ytr),
    "OLS test": fits["OLS"].score(Xte, yte),
    "Ridge test": fits["Ridge"].score(Xte, yte),
    "Lasso test": fits["Lasso"].score(Xte, yte),
})
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(r2_bars.index, r2_bars.values, color=["#1B2A35", "#64748B", "#64748B", "#0F766E", "#E4572E"])
for k, v in enumerate(r2_bars.values):
    ax.text(k, v + 0.008, f"{v:.3f}", ha="center")
ax.axhline(r2_max, color="#1B2A35", ls=":", lw=1)
ax.set(ylim=(0.5, 1.05), ylabel="R²", title="R² față de maximul posibil")
plt.xticks(rotation=15)
plt.show()



R² pe antrenare al lui OLS (≈ 0,992) este **peste maximul teoretic** (≈ 0,938): imposibil dacă modelul ar fi învățat doar semnal. Restul este zgomot memorat. Se potrivește chiar cu estimarea simplă: reziduurile OLS sunt, în medie, $\sigma^2 \cdot$ (grade de libertate) $= 4$, față de o variație totală de ≈ 488, deci $R^2 \approx 0{,}992$.

#### 5.2.2 Ce fac Ridge și Lasso cu variabilele irelevante (slide 35)

Ultimele 22 de variabile ar trebui să aibă coeficientul 0. Cât de departe sunt de 0 coeficienții estimați?


In [ ]:

irel = slice(3, None)                                  # ultimele 22 de variabile (irelevante)
tabel_irel = pd.DataFrame({
    nume: {"max |coef|": np.abs(m.coef_[irel]).max(), "medie |coef|": np.abs(m.coef_[irel]).mean(),
           "coef. ≠ 0 (din 22)": int((m.coef_[irel] != 0).sum())}
    for nume, m in fits.items()}).T

tabel_reale = pd.DataFrame({"adevăr": beta_true[:3], **{nume: m.coef_[:3] for nume, m in fits.items()}},
                           index=["β₁", "β₂", "β₃"]).T

print("Cele 22 de variabile irelevante (adevărul: 0):")
display(tabel_irel.round(2))
print("Cele 3 variabile reale:")
display(tabel_reale.round(2))
print(f"Lasso: {int((fits['Lasso'].coef_ != 0).sum())} coeficienți nenuli = "
      f"{int((fits['Lasso'].coef_[:3] != 0).sum())} reali + {int((fits['Lasso'].coef_[irel] != 0).sum())} falși; "
      f"{int((fits['Lasso'].coef_ == 0).sum())} sunt exact 0")

tabel_irel[["max |coef|", "medie |coef|"]].plot.bar(figsize=(7, 4), color=["#64748B", "#7C6AA6"], rot=0,
                                                    title="Cele 22 de variabile irelevante (adevărul: 0)")
plt.show()



- **OLS** dă coeficienți mari variabilelor irelevante: a „găsit" tipare în zgomot.
- **Ridge** îi micșorează, dar toți cei 25 rămân nenuli.
- **Lasso** păstrează 8 coeficienți nenuli (cei 3 reali și 5 falși); ceilalți 17 sunt exact 0.
- **Prețul regularizării este bias-ul:** coeficientul real 1,5 este estimat ≈ 0,95 de OLS, ≈ 0,93 de Ridge și ≈ 0,66 de Lasso. Regularizarea câștigă pe test prin reducerea varianței, nu prin estimări mai „corecte".

#### 5.2.3 Concluzie: decalajul antrenare → test (slide 36)


In [ ]:

concluzie = pd.DataFrame({
    nume: {"R² antrenare": m.score(Xtr, ytr), "R² test": m.score(Xte, yte),
           "decalaj (antrenare − test)": m.score(Xtr, ytr) - m.score(Xte, yte),
           "RMSE test": rmse(yte, m.predict(Xte)), "coef. ≠ 0": int((m.coef_ != 0).sum())}
    for nume, m in fits.items()}).T
concluzie.round(3)



Decalajul dintre $R^2$ pe antrenare și $R^2$ pe test este cel mai mare la OLS (≈ 0,23) și cel mai mic la Lasso (≈ 0,10). Ridge reduce varianța, dar păstrează toate variabilele. Lasso folosește faptul că adevărul este **rar** (3 variabile din 25) și are cea mai bună performanță pe test.

De reținut: (1) este o **singură rulare**: valorile variază, ordinea rămâne; (2) regularizarea schimbă varianță mai mică pe bias; (3) cine câștigă depinde de structura problemei (vezi 5.3).

### 5.3 Ridge sau Lasso? Depinde de structura problemei (slide 37)

Comparăm OLS, Ridge și Lasso în trei scenarii cu $p = 25$ variabile independente, $n = 50$ de observații și $\sigma = 1$: adevăr **rar** (3 variabile contează), **intermediar** (8) și **dens** (toate 25 contează puțin, $\pm 0{,}6$). Raportăm RMSE pe un test de 4.000 de observații, mediat pe 100 de repetări. Minimul posibil este 1,00 (chiar $\sigma$). *Celula durează 1–2 minute; micșorați `REPS` pentru o rulare rapidă.*


In [ ]:

import warnings
rng_struct = np.random.default_rng(7)                 # generator propriu
p_s, n_s, REPS, N_TEST = 25, 50, 100, 4000

scenarii = {}
b = np.zeros(p_s); b[:3] = [3, -2, 1.5]
scenarii["Rar: 3 din 25"] = b
b = rng_struct.choice([-1, 1], size=p_s) * 0.6
scenarii["Dens: toate 25"] = b
b = np.zeros(p_s); b[:8] = rng_struct.choice([-1, 1], size=8) * 1.6
scenarii["Intermediar: 8 din 25"] = b

rezultate, nenule = {}, {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                   # avertismente de convergență la Lasso, nu afectează rezultatul
    for nume, beta_s in scenarii.items():
        rm = {"OLS": [], "Ridge": [], "Lasso": []}; nz = []
        for _ in range(REPS):
            Xs_ = rng_struct.normal(size=(n_s, p_s)); ys_ = Xs_ @ beta_s + rng_struct.normal(size=n_s)
            Xt_ = rng_struct.normal(size=(N_TEST, p_s)); yt_ = Xt_ @ beta_s + rng_struct.normal(size=N_TEST)
            for k, m in {"OLS": LinearRegression(), "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 40)),
                         "Lasso": LassoCV(alphas=np.logspace(-3, 1, 40), cv=5, max_iter=20000)}.items():
                m.fit(Xs_, ys_)
                rm[k].append(rmse(yt_, m.predict(Xt_)))
                if k == "Lasso":
                    nz.append(int((m.coef_ != 0).sum()))
        rezultate[nume] = {k: np.mean(v) for k, v in rm.items()}
        nenule[nume] = np.mean(nz)

struct_df = pd.DataFrame(rezultate).T.loc[["Rar: 3 din 25", "Intermediar: 8 din 25", "Dens: toate 25"]]
struct_df["Lasso: coef. ≠ 0 (medie)"] = pd.Series(nenule)
ax = struct_df[["OLS", "Ridge", "Lasso"]].plot.bar(figsize=(8, 4.5), color=["#64748B", "#0F766E", "#E4572E"], rot=0)
ax.axhline(1.0, color="black", ls=":", lw=1)
ax.set(ylim=(1.0, 1.6), ylabel="RMSE pe test (minim posibil = 1,00)", title="Ridge sau Lasso: depinde de structura adevărului")
plt.show()
struct_df.round(2)



- **Adevăr rar → Lasso:** elimină variabilele irelevante (coeficienți exact 0) și scapă de zgomot cu puțin bias. Aici ≈ 1,13 față de ≈ 1,42 la Ridge.
- **Efecte multe și mici → Ridge:** păstrează toate variabilele și le micșorează. Lasso ar elimina variabile care contribuie real și iese chiar mai prost decât OLS (≈ 1,47).
- **Structura nu o știm dinainte:** comparați modelele prin validare încrucișată sau folosiți ElasticNet. „Rar" înseamnă, în practică, „aproape rar": puține variabile explică cea mai mare parte din semnal.


## 6. Compromisul bias–varianță și alegerea lui $\alpha$

### 6.1 Compromisul bias–varianță (slide 39)

Eroarea de predicție pe date noi se descompune, pentru un punct dat, în trei părți:

$$E[(y - \hat y)^2] = \underbrace{\text{Bias}^2}_{\text{modelul e prea simplu}} + \underbrace{\text{Varianța}}_{\text{modelul depinde prea mult de eșantion}} + \underbrace{\sigma^2}_{\text{zgomot ireductibil}}$$

- OLS este **nedeplasat** (bias ≈ 0), dar poate avea **varianță mare**.
- Regularizarea introduce **intenționat un mic bias** (coeficienții sunt trași spre zero) în schimbul unei **scăderi mai mari a varianței**. Dacă scăderea varianței depășește creșterea lui bias², eroarea totală scade.
- Parametrul $\alpha$ reglează balanța: $\alpha = 0$ înseamnă OLS (bias minim, varianță maximă); $\alpha \to \infty$ înseamnă model care prezice doar media (bias mare, varianță zero).

In [ ]:
p2, sigma = 30, 3.0
beta2 = np.zeros(p2)
beta2[:5] = [3, -2, 1.5, 1, -1]

def make2(n):
    Xn = rng.normal(size=(n, p2))
    return Xn, Xn @ beta2 + sigma * rng.normal(size=n)

Xa, ya = make2(60)
Xb, yb = make2(5000)

grids = {"Ridge": np.logspace(-2, 4, 40), "Lasso": np.logspace(-3, 1, 40)}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (name, Est) in zip(axes, {"Ridge": Ridge, "Lasso": Lasso}.items()):
    tr, te = [], []
    for a in grids[name]:
        m = Est(alpha=a, max_iter=50000).fit(Xa, ya) if name == "Lasso" else Est(alpha=a).fit(Xa, ya)
        tr.append(mean_squared_error(ya, m.predict(Xa)))
        te.append(mean_squared_error(yb, m.predict(Xb)))
    ax.plot(grids[name], tr, label="eroare pe train")
    ax.plot(grids[name], te, label="eroare pe test")
    best = grids[name][int(np.argmin(te))]
    ax.axvline(best, color="red", ls="--", label=f"minimul testului: α ≈ {best:.3g}")
    ax.axhline(sigma**2, color="gray", ls=":", label="zgomot ireductibil (σ²)")
    ax.set(xscale="log", xlabel="alpha", ylabel="MSE", title=name)
    ax.legend()
plt.tight_layout()
plt.show()

Eroarea pe **train** crește monoton cu $\alpha$ (un model mai constrâns se potrivește mai prost pe datele văzute), în timp ce eroarea pe **test** are formă de „U”: scade mai întâi (varianța se reduce), apoi crește (bias-ul domină). Minimul este $\alpha$-ul pe care vrem să-l găsim, iar fără set de test îl căutăm prin validare încrucișată (vezi 6.6).


### 6.2 Cele trei componente ale erorii (slide 40)

| | Bias² | Varianță | Zgomot $\sigma^2$ |
|---|---|---|---|
| **Ce măsoară** | cât de departe este, în medie, predicția modelului de adevăr | cât s-ar schimba predicția dacă antrenăm pe alt eșantion | partea imprevizibilă din $y$ (aici $\sigma = 3$, deci $\sigma^2 = 9$) |
| **Când este mare** | model prea simplu sau prea „strâns": $\alpha$ mare, variabile importante eliminate | model flexibil, puține date, variabile corelate: $\alpha$ mic | nu depinde de model |
| **Cum o reduci** | mai multă flexibilitate: $\alpha$ mai mic, mai multe variabile | regularizare ($\alpha$ mai mare), mai multe date, mai puține variabile | nu se poate reduce: este limita de jos a erorii |

Rolul lui $\alpha$: $\alpha = 0$ (OLS) înseamnă bias minim și varianță maximă; $\alpha$ optim este compromisul; $\alpha \to \infty$ înseamnă un model care prezice doar media (varianță 0, bias mare).

### 6.3 Bias și varianță, măsurate (slide 41)

Măsurăm efectiv cele trei componente pentru Ridge. Antrenăm modelul pe **300 de seturi de antrenare diferite** (60 de observații, 30 de variabile, $\sigma = 3$) și, pe puncte de test fixe, calculăm pentru fiecare $\alpha$:

- **Bias²** = media pe puncte a lui (predicția medie − valoarea adevărată)²;
- **Varianța** = media pe puncte a varianței predicțiilor între cele 300 de seturi;
- eroarea totală așteptată = Bias² + Varianță + $\sigma^2$.


In [ ]:

rng_d = np.random.default_rng(2024)                   # generator propriu
p_d, n_d, sigma_d, R = 30, 60, 3.0, 300
beta_d = np.zeros(p_d); beta_d[:5] = [3, -2, 1.5, 1, -1]
X_fix = rng_d.normal(size=(1000, p_d))                 # puncte de test fixe
f_fix = X_fix @ beta_d                                 # valorile adevărate (fără zgomot)
alphas_d = np.logspace(-2, 4, 13)

pred = {a: [] for a in alphas_d}
for _ in range(R):                                     # 300 de seturi de antrenare diferite
    Xt_ = rng_d.normal(size=(n_d, p_d)); yt_ = Xt_ @ beta_d + sigma_d * rng_d.normal(size=n_d)
    for a in alphas_d:
        pred[a].append(Ridge(alpha=a).fit(Xt_, yt_).predict(X_fix))

rows = []
for a in alphas_d:
    P = np.array(pred[a])                              # (R, 1000)
    bias2 = float(np.mean((P.mean(axis=0) - f_fix) ** 2))
    var_ = float(P.var(axis=0).mean())
    rows.append({"α": a, "Bias²": bias2, "Varianță": var_, "σ²": sigma_d**2, "Total = Bias² + Var + σ²": bias2 + var_ + sigma_d**2})
decomp = pd.DataFrame(rows).set_index("α")

fig, ax = plt.subplots(figsize=(9, 5))
for col, color in [("Bias²", "#E4572E"), ("Varianță", "#0F766E"), ("σ²", "#D99A00"), ("Total = Bias² + Var + σ²", "#1B2A35")]:
    ax.plot(decomp.index, decomp[col], marker="o", ms=5, lw=2.5, label=col, color=color)
alpha_min = decomp["Total = Bias² + Var + σ²"].idxmin()
ax.axvline(alpha_min, color="gray", ls="--", lw=1)
ax.set(xscale="log", xlabel="α (scară logaritmică)", ylabel="eroare pe test",
       title=f"Descompunerea erorii pe test, Ridge (medie pe {R} seturi de antrenare); minim la α = {alpha_min:.3g}")
ax.legend()
plt.show()
decomp.round(2)



Bias² crește cu $\alpha$, varianța scade, iar suma plus $\sigma^2$ are minim la $\alpha \approx 10$. La $\alpha = 0{,}01$ (≈ OLS): Bias² ≈ 0,03 și varianța ≈ 9,61; la $\alpha = 10$: Bias² ≈ 0,71 și varianța ≈ 4,51 (total ≈ 14,22, minimul); la $\alpha = 10.000$: Bias² ≈ 15,70 și varianța ≈ 0,47.

> **Atenție.** Curba de test din celula din 6.1 (și din slide-ul 39) vine dintr-**un singur** set de antrenare; aici avem **media pe 300** de seturi, deci estimarea erorii *așteptate*. De aceea valorile sunt puțin mai mari aici (≈ 14,22 față de ≈ 12,91 la $\alpha = 10$).

### 6.4 Trei regimuri: supraînvățare, compromis, subpotrivire (slide 39 și 42)

Reproducem curba din slide-ul 39, **cu linia zgomotului $\sigma^2 = 9$**, și extragem cele trei regimuri: $\alpha$ mic (supraînvățare), $\alpha$ optim (compromis) și $\alpha$ mare (subpotrivire).


In [ ]:

rng_b = np.random.default_rng(42)                      # generator propriu
p_b, sigma_b = 30, 3.0
beta_b = np.zeros(p_b); beta_b[:5] = [3, -2, 1.5, 1, -1]

def make_b(n):
    Xn = rng_b.normal(size=(n, p_b))
    return Xn, Xn @ beta_b + sigma_b * rng_b.normal(size=n)

Xa_b, ya_b = make_b(60)                                # un singur set de antrenare
Xb_b, yb_b = make_b(5000)                              # test mare
g = np.logspace(-2, 4, 25)
tr_b = np.array([mean_squared_error(ya_b, Ridge(alpha=a).fit(Xa_b, ya_b).predict(Xa_b)) for a in g])
te_b = np.array([mean_squared_error(yb_b, Ridge(alpha=a).fit(Xa_b, ya_b).predict(Xb_b)) for a in g])
i_best = int(np.argmin(te_b))

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(g, tr_b, color="#64748B", lw=2.5, label="eroare pe antrenare")
ax.plot(g, te_b, color="#E4572E", lw=2.5, label="eroare pe test")
ax.axhline(sigma_b**2, color="#D99A00", lw=2.5, label="zgomot ireductibil (σ² = 9)")
ax.axvline(g[i_best], color="gray", ls="--", lw=1, label=f"minimul testului: α ≈ {g[i_best]:.3g}")
ax.set(xscale="log", xlabel="α (scară logaritmică)", ylabel="MSE", title="Ridge: MSE în funcție de α")
ax.legend()
plt.show()

var_y_b = sigma_b**2 + float(beta_b @ beta_b)           # varianța lui y: 9 + 17,25 = 26,25
regimuri = pd.DataFrame({
    "α": [g[0], g[i_best], g[-1]],
    "eroare antrenare": [tr_b[0], tr_b[i_best], tr_b[-1]],
    "eroare test": [te_b[0], te_b[i_best], te_b[-1]],
}, index=["α mic: supraînvățare", "α optim: compromis", "α mare: subpotrivire"])
regimuri["decalaj test − antrenare"] = regimuri["eroare test"] - regimuri["eroare antrenare"]
print(f"Câștig la α optim față de α mic: {100 * (te_b[0] - te_b[i_best]) / te_b[0]:.1f}% | varianța lui y = {var_y_b:.2f}")
regimuri.round(2)



- **$\alpha$ mic (≈ OLS): supraînvățare.** Eroare pe antrenare ≈ 4,14, pe test ≈ 14,66: decalaj ≈ 10,5. Varianță mare, modelul memorează zgomotul.
- **$\alpha \approx 10$: compromis.** Eroare pe antrenare ≈ 4,71, pe test ≈ 12,91 (minim, cu ≈ 12% mai puțin decât la $\alpha$ mic); decalaj ≈ 8,2.
- **$\alpha$ mare (10.000): subpotrivire.** ≈ 27,04 pe antrenare și ≈ 26,84 pe test: decalaj ≈ 0, dar modelul prezice doar media, iar eroarea ≈ varianța lui $y$ (26,25).

**Atenție:** decalajul dintre antrenare și test scade mereu cu $\alpha$, dar eroarea pe test nu. Contează **minimul erorii pe test**, nu decalajul zero.

### 6.5 Concluzie: bias–varianță (slide 43)

1. **Regularizarea schimbă varianță pe bias.** Cu $\alpha$ crește bias-ul și scade varianța; eroarea totală este minimă unde cele două se echilibrează (aici $\alpha \approx 10$).
2. **$\alpha$ nu se alege pe antrenare.** Eroarea pe antrenare crește mereu cu $\alpha$ și ar alege $\alpha = 0$. Se alege prin validare încrucișată (vezi 6.6).
3. **Zgomotul pune o limită.** Nicio valoare a lui $\alpha$ nu coboară sub $\sigma^2 = 9$: câștigul posibil este doar peste această limită.

*De reținut:* câștigul este moderat (≈ 12% aici), valorile variază de la un eșantion la altul, dar forma de „U" a erorii pe test se păstrează.


### 6.6 Alegerea lui $\alpha$ (slide-urile 44–46)

$\alpha$ **nu se învață din date prin minimizarea erorii de antrenare** (aceasta ar alege mereu $\alpha = 0$). Se alege prin **validare încrucișată**: pentru fiecare valoare dintr-o grilă se calculează eroarea medie pe foldurile de validare, apoi se alege valoarea cu eroarea cea mai mică (`RidgeCV`, `LassoCV`, `ElasticNetCV`).

**Reguli practice:**
1. **Grilă logaritmică** (`np.logspace`): efectul lui $\alpha$ se vede la ordine de mărime.
2. **Intervalul trebuie să încadreze minimul.** Dacă $\alpha$ ales este chiar la **marginea grilei**, minimul real e probabil dincolo: extindeți grila.
3. **Scara contează:** $\alpha$ depinde de scara lui $y$ (și, la Ridge, de $n$). Valori „tipice” dintr-un alt set de date nu se transferă.
4. **Regula „o eroare standard” (1-SE):** în loc de $\alpha$ cu eroarea minimă, se ia cel mai **mare** $\alpha$ a cărui eroare este în intervalul $\text{minim} \pm 1\,\text{SE}$. Se obține un model mai simplu (mai puține variabile) cu performanță statistic echivalentă.
5. Standardizarea se face **în interiorul** validării (pipeline), nu o singură dată pe tot setul, ca să nu apară *leakage*.

In [ ]:
a_grid = np.logspace(-2, 2, 80)
pipe_l = make_pipeline(StandardScaler(), LassoCV(alphas=a_grid, cv=cv, max_iter=50000))
pipe_l.fit(X_train, y_train)
lcv = pipe_l[-1]

mean_mse = lcv.mse_path_.mean(axis=1)
se_mse = lcv.mse_path_.std(axis=1, ddof=1) / np.sqrt(cv.get_n_splits())
i_min = int(np.argmin(mean_mse))
alpha_min = lcv.alphas_[i_min]
alpha_1se = lcv.alphas_[mean_mse <= mean_mse[i_min] + se_mse[i_min]].max()   # regula „o eroare standard”

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(lcv.alphas_, mean_mse)
ax.fill_between(lcv.alphas_, mean_mse - se_mse, mean_mse + se_mse, alpha=0.25)
ax.axvline(alpha_min, color="red", ls="--", label=f"alpha minim = {alpha_min:.2f}")
ax.axvline(alpha_1se, color="green", ls="--", label=f"alpha „1-SE” = {alpha_1se:.2f}")
ax.set(xscale="log", xlabel="alpha", ylabel="MSE în validare încrucișată", title="Alegerea lui alpha cu LassoCV")
ax.legend()
plt.show()

Linia roșie este $\alpha$ cu eroarea minimă, iar cea verde este alegerea 1-SE (mai mare, deci mai multă regularizare). Banda colorată arată incertitudinea (± o eroare standard) a erorii de validare: pe un interval larg de valori ale lui $\alpha$ diferențele dintre modele sunt de același ordin cu zgomotul, de aceea regula 1-SE este rezonabilă.

## 7. Aplicație completă pe `diabetes`

Comparăm OLS, Ridge, Lasso și ElasticNet, cu pipeline (standardizare + model), folosind aceeași împărțire train/test.

In [ ]:
def make_pipe(estimator):
    """Standardizarea face parte din pipeline: se învață doar pe datele de antrenare."""
    return make_pipeline(StandardScaler(), estimator)

models = {
    "OLS": make_pipe(LinearRegression()),
    "Ridge": make_pipe(RidgeCV(alphas=np.logspace(-3, 3, 50))),
    "Lasso": make_pipe(LassoCV(alphas=np.logspace(-2, 2, 80), cv=cv, max_iter=50000)),
    "ElasticNet": make_pipe(ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], alphas=np.logspace(-2, 2, 80), cv=cv, max_iter=50000)),
}

rows = {}
for name, m in models.items():
    cv_rmse = -cross_val_score(m, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    m.fit(X_train, y_train)
    est = m[-1]
    pred = m.predict(X_test)
    rows[name] = {
        "RMSE CV": cv_rmse.mean(), "RMSE test": rmse(y_test, pred), "R² test": r2_score(y_test, pred),
        "coef. ≠ 0": int((est.coef_ != 0).sum()), "alpha ales": getattr(est, "alpha_", np.nan),
    }
results = pd.DataFrame(rows).T
results.round(3)

Coeficienții (pe variabile standardizate, deci comparabili între ei):

In [ ]:
coef_df = pd.DataFrame({name: m[-1].coef_ for name, m in models.items()}, index=X.columns)
coef_df.round(2)

**Cum interpretăm.**
- Priviți coloanele `s1`, `s2` (corelate la ≈ 0,9): la OLS au valori mari și de semne opuse (în rularea noastră ≈ −44 și ≈ +25), la Ridge sunt micșorate moderat (≈ −34 și ≈ +16; $\alpha$ ales este mic). Lasso este cel mai „agresiv”: `s2` devine exact 0, iar `s1` scade la ≈ −11. ElasticNet, cu un $\alpha$ foarte mic (≈ 0,03), rămâne aproape de Ridge (≈ −32 și ≈ +15) și nu elimină nicio variabilă.
- Lasso păstrează 9 din cele 10 variabile (doar `s2` este eliminată: `s1` o „acoperă”, fiind corelate la ≈ 0,9). Variabilele `bmi`, `s5` și `bp` rămân importante în toate modelele.
- Diferențele de eroare între modele sunt **mici** (RMSE pe test ≈ 53–54 pentru toate patru): la 442 de observații și 10 variabile, OLS nu supraînvață dramatic. Beneficiul principal aici este modelul mai **stabil și mai ușor de interpretat**, mai puțin un câștig mare de acuratețe. Acesta este și motivul pentru care în LP5 și LP6 (mii de observații, puține variabile) cele trei metode au ieșit aproape identice: regularizarea contează mai mult când $n$ este mic față de $p$ sau când variabilele sunt foarte corelate.

## 8. Capcane și rezumat

**Capcane frecvente**
1. **Fără standardizare** penalizarea tratează inegal variabilele (alege scara, nu importanța). Standardizați în pipeline.
2. **Standardizare pe tot setul înainte de CV/split** → *leakage*.
3. **Interceptul nu se penalizează** (scikit-learn se ocupă singur de asta).
4. **Coeficienții sunt deplasați (biased).** Nu-i interpretați ca efecte „adevărate”. În plus, **p-values și intervalele de încredere clasice (`statsmodels` OLS) nu sunt valabile** pe modele penalizate.
5. **Selecția Lasso ≠ importanță cauzală**; variabila eliminată poate fi doar „acoperită” de una corelată.
6. **Alpha la marginea grilei** → extindeți grila.
7. **Compararea valorilor lui `alpha`** între Ridge și Lasso nu are sens (definiții diferite).
8. Penalizarea se aplică și variabilelor one-hot: alegerea categoriei de referință și scara afectează rezultatul; verificați rezultatul cu atenție.

**Rezumat într-o frază.** Ridge și Lasso adaugă o penalizare pe mărimea coeficienților: Ridge îi micșorează fără să-i anuleze și stabilizează variabilele corelate, Lasso îi poate anula și face selecție de variabile, iar $\alpha$ se alege prin validare încrucișată pe o grilă logaritmică, după standardizarea variabilelor în pipeline.

## 9. Exerciții individuale

### ✍️ Exercițiul 1 ⭐ – Calea Ridge pe `diabetes`
Folosind `Xs` (variabilele standardizate) și `y`:
1. Ajustați `Ridge` pentru o grilă `np.logspace(-2, 4, 60)` și rețineți coeficienții.
2. Reprezentați căile coeficienților.
3. Care variabilă se micșorează cel mai repede când $\alpha$ crește? Care își schimbă semnul (dacă există) pe parcurs? Explicați cu ajutorul corelațiilor dintre `s1`, ..., `s6` (`X.corr()`).

In [ ]:
# Codul vostru aici

### ✍️ Exercițiul 2 ⭐ – Ce se întâmplă fără standardizare?
1. Creați o copie a lui `X_train` și `X_test` în care coloana `age` este înmulțită cu 1000 (ca și cum ar fi măsurată în alte unități).
2. Ajustați `Lasso(alpha=5.0)` **cu** `StandardScaler` (pipeline) și **fără** el, pe datele modificate. Comparați variabilele cu coeficient ≠ 0.
3. Explicați de ce `age` și `s5` (variabila cu efectul cel mai puternic, dar cu valori mici, în jur de 4–5) sunt tratate diferit în cele două cazuri.

In [ ]:
# Codul vostru aici

### ✍️ Exercițiul 3 ⭐⭐ – Marginea grilei
1. Rulați `RidgeCV(alphas=np.logspace(-4, -2, 20))` într-un pipeline cu standardizare, pe `X_train`, `y_train`. Cât este `alpha_`? Este la marginea grilei?
2. Extindeți grila (`np.logspace(-4, 4, 60)`) și repetați. Cât devine `alpha_` acum?
3. Calculați, pentru fiecare $\alpha$ din grila extinsă, eroarea medie (RMSE) din validarea încrucișată cu 5 foldy (`cross_val_score` într-o buclă) și reprezentați curba. Marcați minimul.

In [ ]:
# Codul vostru aici

### ✍️ Exercițiul 4 ⭐⭐ – Stabilitatea selecției
Folosind `alpha_` ales de `models["Lasso"]` (și `alpha_`, `l1_ratio_` ale lui `models["ElasticNet"]`):
1. Repetați de 100 de ori: extrageți un eșantion *bootstrap* din setul de antrenare (cu repetiție, `X_train.sample(frac=1, replace=True, random_state=i)`), ajustați Lasso cu `alpha` fix (în pipeline cu standardizare) și notați care variabile au coeficient ≠ 0.
2. Calculați frecvența de selecție a fiecărei variabile. Care sunt selectate mereu? Care sunt instabile?
3. Repetați cu ElasticNet (parametrii fixați). Selecția este mai stabilă pentru variabilele corelate (`s1`, `s2`)?

In [ ]:
# Codul vostru aici

### ✍️ Exercițiul 5 ⭐⭐⭐ – Mai multe variabile decât observații ($p > n$)
Generați `n = 40`, `p = 100`, cu doar 5 variabile cu efect real (`beta[:5] = [4, -3, 2, -2, 1.5]`), zgomot standard normal și un set de test de 5.000 de observații.
1. Ajustați `LinearRegression`. Ce R² obțineți pe train și pe test? De ce nu există o soluție unică?
2. Ajustați `RidgeCV` și `LassoCV`. Comparați R² pe test.
3. Pe calea Lasso (`lasso_path`), pentru fiecare $\alpha$ calculați câte din cele 5 variabile reale au fost găsite (*recall*) și câte variabile false au fost selectate. Reprezentați ambele curbe în funcție de $\alpha$. Există un $\alpha$ la care găsiți toate cele 5 fără variabile false? Cât de multe false apar la cel mai mare $\alpha$ la care sunt încă găsite toate cele 5?

In [ ]:
# Codul vostru aici

### ✍️ Exercițiul 6 ⭐⭐⭐ – Regula „o eroare standard”
Pornind de la `alpha_min` și `alpha_1se` din secțiunea 6:
1. Ajustați două modele Lasso (în pipeline cu standardizare) cu cele două valori ale lui $\alpha$ pe `X_train`.
2. Comparați RMSE pe test și numărul de coeficienți ≠ 0.
3. Ce alegeți dacă prioritatea este un model simplu și interpretabil? Dar dacă este acuratețea pură?

In [ ]:
# Codul vostru aici

### ✍️ Exercițiul 7 ⭐⭐⭐ – De ce în LP6 cele trei modele au ieșit identice?
Reluați setul `diamonds` din LP6 (`sns.load_dataset("diamonds")`, curățat ca acolo: fără dimensiuni zero, `y < 20`, `z < 20`). Folosiți variabilele `log(carat)`, `log(x)`, `log(y)`, `log(z)`, `depth`, `table` și ținta `log(price)`.
1. Păstrați un set de test fix de 2.000 de diamante. Pentru `n_train` în `[30, 60, 150, 1000, 5000]`, repetați de 20 de ori: extrageți aleator `n_train` diamante pentru antrenare, ajustați OLS, `RidgeCV` și `LassoCV` (cu standardizare) și calculați RMSE pe test (pe scara log).
2. Reprezentați RMSE mediu în funcție de `n_train` pentru cele trei modele (axa x logaritmică).
3. Concluzie: când contează regularizarea? Legați răspunsul de raportul $n/p$ și de multicoliniaritatea dintre `carat`, `x`, `y`, `z`.

In [ ]:
# Codul vostru aici

## Întrebări de verificare

1. De ce $X^\top X + \alpha I$ este inversabilă pentru $\alpha > 0$, chiar dacă $X^\top X$ nu este?
2. Ce reprezintă „bias” și „varianță” în contextul regularizării? De ce acceptăm un bias mai mare?
3. Explicați cu cuvintele voastre (geometric sau algebric) de ce Lasso poate anula coeficienți, iar Ridge nu.
4. De ce trebuie standardizate variabilele înainte de Ridge/Lasso? De ce nu se penalizează interceptul?
5. Ce înseamnă că $\alpha$ ales este „la marginea grilei” și ce faceți atunci?
6. Ce avantaj are regula 1-SE? Ce pierdeți aplicând-o?
7. Când ați alege ElasticNet în loc de Lasso?
8. De ce nu putem interpreta p-values din `statsmodels` pentru un model Lasso?